# Image Resizing for Vision Model Fine-Tuning

This notebook prepares a lighter-weight version of the raw phone images for model fine-tuning.

Goals:
- Reduce file size (MB -> KB) while preserving visual quality.
- Standardize spatial resolution for training (e.g. 224x224 or 384x384, configurable).
- Keep original files intact; write resized copies to `resized_images/` (mirrors structure if subfolders added later).
- Optionally strip metadata (EXIF) to save space and avoid orientation issues.
- Handle motion-photo `.MP.jpg` files the same as normal JPEGs.

After running, you'll have uniformly sized images suitable for fast dataloader throughput.

Configure parameters in the next cell, then run the processing function cell, then the summary.

If you want different sizes for experimentation, just re-run with another OUTPUT_SUBDIR (e.g. `resized_384`).

In [7]:
# Configuration & Imports
from pathlib import Path
from PIL import Image, ImageOps
import os, io, math, statistics, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Tuple, Dict

# ---- Parameters ----
BASE_DIR = Path('images')              # source directory with original photos
OUTPUT_ROOT = Path('resized_images')   # root directory to store resized sets
TARGET_SIZE = (224, 224)               # (width, height)
MAINTAIN_ASPECT = True                 # If True, keep aspect & pad to square; else direct resize
PADDING_COLOR = (0, 0, 0)              # RGB padding color when maintaining aspect
MAX_WORKERS = 8                        # Thread pool size for IO-bound PIL operations
QUALITY = 88                           # JPEG quality (trade-off size vs quality)
PROGRESS_EVERY = 25                    # Print progress every N images
DRY_RUN = False                         # If True, just simulate & collect stats
OVERWRITE = False                      # If False skip if output exists
STRIP_METADATA = True                  # Remove EXIF to reduce size & orientation headaches
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
OUTPUT_SUBDIR = f"{TARGET_SIZE[0]}x{TARGET_SIZE[1]}"  # e.g. 224x224 for separate experiments

OUTPUT_DIR = OUTPUT_ROOT / OUTPUT_SUBDIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input dir: {BASE_DIR.resolve()}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Target size: {TARGET_SIZE}  Maintain aspect: {MAINTAIN_ASPECT}")

Input dir: D:\Projects\DS&ML\Futury_AI_Hackathon\images
Output dir: D:\Projects\DS&ML\Futury_AI_Hackathon\resized_images\224x224
Target size: (224, 224)  Maintain aspect: True


In [8]:
# Processing Functions & Execution
from time import perf_counter

def is_image_file(p: Path) -> bool:
    return p.is_file() and p.suffix.lower() in VALID_EXTS

def list_images(base: Path) -> List[Path]:
    return [f for f in base.iterdir() if is_image_file(f)]

def open_image(path: Path):
    try:
        img = Image.open(path)
        img.load()  # force load
        if img.mode not in ('RGB', 'RGBA'):  # normalize
            img = img.convert('RGB')
        return img
    except Exception as e:
        print(f"[WARN] Failed to open {path.name}: {e}")
        return None

def resize_and_pad(img: Image.Image, size: Tuple[int,int]) -> Image.Image:
    if not MAINTAIN_ASPECT:
        return img.resize(size, Image.Resampling.LANCZOS)
    tw, th = size
    iw, ih = img.size
    scale = min(tw/iw, th/ih)
    nw, nh = int(iw*scale), int(ih*scale)
    img = img.resize((nw, nh), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', size, PADDING_COLOR)
    canvas.paste(img, ((tw-nw)//2, (th-nh)//2))
    return canvas

def save_image(img: Image.Image, dest: Path):
    params = {}
    fmt = 'JPEG'
    params['quality'] = QUALITY
    params['optimize'] = True
    params['progressive'] = True
    if STRIP_METADATA and 'exif' in img.info:
        params['exif'] = b''  # strip
    dest.parent.mkdir(parents=True, exist_ok=True)
    img.save(dest, format=fmt, **params)

def process_one(path: Path) -> Dict:
    out_path = OUTPUT_DIR / path.name
    if out_path.exists() and not OVERWRITE:
        return {'name': path.name, 'skipped': True, 'orig_bytes': path.stat().st_size, 'new_bytes': out_path.stat().st_size}
    img = open_image(path)
    if img is None:
        return {'name': path.name, 'error': True}
    resized = resize_and_pad(img, TARGET_SIZE)
    if not DRY_RUN:
        save_image(resized, out_path)
    new_bytes = out_path.stat().st_size if out_path.exists() else 0
    return {
        'name': path.name,
        'orig_bytes': path.stat().st_size,
        'new_bytes': new_bytes,
        'skipped': False,
        'error': False
    }

start = perf_counter()
images = list_images(BASE_DIR)
print(f"Found {len(images)} candidate images.")

# Limit for dry run preview
if DRY_RUN:
    images = images[:min(10, len(images))]
    print(f"DRY_RUN active: limiting to {len(images)} images.")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    fut_map = {ex.submit(process_one, p): p for p in images}
    for i, fut in enumerate(as_completed(fut_map)):
        res = fut.result()
        results.append(res)
        if (i+1) % PROGRESS_EVERY == 0 or (i+1) == len(images):
            print(f"Processed {i+1}/{len(images)}")

elapsed = perf_counter() - start
print(f"Done in {elapsed:.2f}s")

# Aggregate stats
processed = [r for r in results if not r.get('error')]
orig_sizes = [r.get('orig_bytes',0) for r in processed]
new_sizes = [r.get('new_bytes',0) for r in processed if not DRY_RUN]

if orig_sizes:
    avg_orig = statistics.mean(orig_sizes)/1024
    print(f"Avg original size: {avg_orig:.1f} KB")
if new_sizes:
    avg_new = statistics.mean(new_sizes)/1024
    reduction_pct = 100 * (1 - (avg_new / (statistics.mean(orig_sizes)/1024)))
    print(f"Avg resized size: {avg_new:.1f} KB  (↓ {reduction_pct:.1f}% vs original)")

# Store summary in a global dict for next cell
SUMMARY = {
    'total_found': len(list_images(BASE_DIR)),
    'processed_count': len(processed),
    'dry_run': DRY_RUN,
    'elapsed_sec': elapsed,
    'target_size': TARGET_SIZE,
    'output_dir': str(OUTPUT_DIR),
    'avg_orig_kb': round(statistics.mean(orig_sizes)/1024,1) if orig_sizes else None,
    'avg_new_kb': round(statistics.mean(new_sizes)/1024,1) if new_sizes else None
}


Found 127 candidate images.
Processed 25/127
Processed 25/127
Processed 50/127
Processed 50/127
Processed 75/127
Processed 75/127
Processed 100/127
Processed 100/127
Processed 125/127
Processed 127/127
Done in 2.84s
Avg original size: 1923.5 KB
Avg resized size: 6.7 KB  (↓ 99.7% vs original)
Processed 125/127
Processed 127/127
Done in 2.84s
Avg original size: 1923.5 KB
Avg resized size: 6.7 KB  (↓ 99.7% vs original)


In [9]:
# Summary
from pprint import pprint
print("Summary of resizing run:")
pprint(SUMMARY)

if SUMMARY.get('dry_run'):
    print("\nDRY_RUN was True: No files were written. Set DRY_RUN=False in the config cell and re-run to save resized images.")

Summary of resizing run:
{'avg_new_kb': 6.7,
 'avg_orig_kb': 1923.5,
 'dry_run': False,
 'elapsed_sec': 2.835549900002661,
 'output_dir': 'resized_images\\224x224',
 'processed_count': 127,
 'target_size': (224, 224),
 'total_found': 127}
